# Data Engineering

# Theme

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, re, subprocess, json, shutil, glob, hashlib
from pathlib import Path
from collections import Counter, defaultdict

COLORS = {"bg":"#F5F5F5","teal":"#76ABAE","dark":"#303841","orange":"#FF5722","white":"#FFFFFF"}
plt.rcParams.update({
    "figure.facecolor": COLORS["bg"], "axes.facecolor": COLORS["bg"],
    "axes.edgecolor": COLORS["dark"], "text.color": COLORS["dark"],
    "axes.labelcolor": COLORS["dark"], "xtick.color": COLORS["dark"], "ytick.color": COLORS["dark"],
    "axes.spines.top": False, "axes.spines.right": False,
})

# Setup

## Folders

In [ ]:
DATA = Path("data")
for sub in ["smartbugs", "swc", "dappscan", "clean_dataset"]:
    (DATA / sub).mkdir(parents=True, exist_ok=True)
print(os.getcwd())

/Users/mk/Desktop/mk/AuditMind/notebooks


## Tracker

In [2]:
class PipelineTracker:
    def __init__(self, name):
        self.name = name
        self.stages = {}
        self.order = []

    def log(self, label, count, detail=None):
        if label not in self.stages:
            self.order.append(label)
        self.stages[label] = (count, detail or {})

    def print_flowchart(self):
        print(f"{'='*50}\n  {self.name}\n{'='*50}")
        for i, label in enumerate(self.order):
            count, detail = self.stages[label]
            count_str = str(count) if count is not None else "--"
            print(f"\n[{count_str}]  {label}")
            for k, v in detail.items():
                print(f"          {v}  {k}")
            if i < len(self.order) - 1:
                print("          |\n          v")

## Solc

In [4]:
PRAGMA_RE = re.compile(r"pragma\s+solidity\s+([^;]+);")
SOLC_PINS = {"0.4":"0.4.26","0.5":"0.5.17","0.6":"0.6.12","0.7":"0.7.6","0.8":"0.8.20"}

def solc_bin(v):
    import sys
    venv_root = os.path.dirname(os.path.dirname(sys.executable))
    return os.path.join(venv_root, ".solc-select", "artifacts", f"solc-{v}", f"solc-{v}")

def resolve_solc_v3(pragma):
    if not pragma:
        return []
    versions = re.findall(r"(\d+)\.(\d+)\.(\d+)", pragma)
    if not versions:
        return []
    mentioned = sorted({(int(a), int(b)) for a, b, _ in versions})
    lo = mentioned[0]
    hi = mentioned[-1] if len(mentioned) > 1 else mentioned[0]
    all_series = sorted(SOLC_PINS.keys())
    in_range = []
    for s in all_series:
        maj, minr = map(int, s.split("."))
        if "<" in pragma:
            if (maj, minr) >= lo and (maj, minr) < hi:
                in_range.append(s)
        else:
            if lo <= (maj, minr) <= hi:
                in_range.append(s)
    if not in_range:
        in_range = [s for s in all_series if (int(s.split(".")[0]), int(s.split(".")[1])) == lo]
    return list(reversed(in_range))

def resolve_solc(pragma):
    candidates = resolve_solc_v3(pragma)
    if not candidates:
        return None, None
    return SOLC_PINS[candidates[0]], candidates[0]

LINE_C = re.compile(r"//[^\n]*")
BLOCK_C = re.compile(r"/\*.*?\*/", re.DOTALL)
def strip_comments(text):
    return LINE_C.sub("", BLOCK_C.sub("", text))

def compile_check(rec, timeout_s=15):
    candidates = resolve_solc_v3(rec.get("pragma")) if rec.get("pragma") else []
    if not candidates:
        return "no_solc_version"
    for series in candidates:
        solc_path = solc_bin(SOLC_PINS[series])
        if not os.path.exists(solc_path):
            continue
        try:
            result = subprocess.run([solc_path, "--bin", rec["path"]],
                                     capture_output=True, text=True, timeout=timeout_s)
        except subprocess.TimeoutExpired:
            continue
        if result.returncode == 0 and "Error" not in result.stderr:
            return "ok"
    return "other"

# SmartBugs

## Pull

In [5]:
smartbugs = PipelineTracker("SmartBugs-curated")

subprocess.run(["curl", "-sL", "--connect-timeout", "30", "--max-time", "180",
    "https://codeload.github.com/smartbugs/smartbugs-curated/tar.gz/refs/heads/main",
    "-o", "smartbugs.tar.gz"], check=True)
subprocess.run(["tar", "-xzf", "smartbugs.tar.gz"], check=True)

all_sol = glob.glob("smartbugs-curated-main/dataset/**/*.sol", recursive=True)
smartbugs.log("Raw files pulled from repo", len(all_sol))
smartbugs.print_flowchart()

  SmartBugs-curated

[143]  Raw files pulled from repo


## Filter

In [6]:
CATEGORIES = ["reentrancy","access_control","denial_of_service",
              "unchecked_low_level_calls","time_manipulation","bad_randomness"]

kept_by_cat = {}
for cat in CATEGORIES:
    src = f"smartbugs-curated-main/dataset/{cat}"
    if os.path.exists(src):
        shutil.copytree(src, DATA/"smartbugs"/cat, dirs_exist_ok=True)
        kept_by_cat[cat] = len([f for f in os.listdir(DATA/"smartbugs"/cat) if f.endswith(".sol")])

smartbugs.log("Dropped 3 irrelevant categories (kept 6 of 10)", None,
              {"arithmetic":"not exploitable post-0.8","front_running":"not source-detectable","other/short_addresses":"catch-all/obsolete"})
smartbugs.log("Filtered to target categories", sum(kept_by_cat.values()), kept_by_cat)
smartbugs.print_flowchart()

  SmartBugs-curated

[143]  Raw files pulled from repo
          |
          v

[--]  Dropped 3 irrelevant categories (kept 6 of 10)
          not exploitable post-0.8  arithmetic
          not source-detectable  front_running
          catch-all/obsolete  other/short_addresses
          |
          v

[120]  Filtered to target categories
          31  reentrancy
          18  access_control
          6  denial_of_service
          52  unchecked_low_level_calls
          5  time_manipulation
          8  bad_randomness


## Pragma

In [7]:
files = []
for cat in CATEGORIES:
    d = DATA/"smartbugs"/cat
    if d.exists():
        for f in d.iterdir():
            if f.suffix == ".sol":
                files.append({"path": str(f), "category": cat})

for rec in files:
    text = open(rec["path"], errors="ignore").read()
    m = PRAGMA_RE.search(text)
    rec["pragma"] = m.group(1).strip() if m else None
    solc, era = resolve_solc(rec["pragma"]) if rec["pragma"] else (None, None)
    rec["solc_version"] = solc
    rec["era"] = era

era_counts = Counter(r["era"] for r in files)
smartbugs.log("Pragma detected", len(files), dict(era_counts))
smartbugs.print_flowchart()

  SmartBugs-curated

[143]  Raw files pulled from repo
          |
          v

[--]  Dropped 3 irrelevant categories (kept 6 of 10)
          not exploitable post-0.8  arithmetic
          not source-detectable  front_running
          catch-all/obsolete  other/short_addresses
          |
          v

[120]  Filtered to target categories
          31  reentrancy
          18  access_control
          6  denial_of_service
          52  unchecked_low_level_calls
          5  time_manipulation
          8  bad_randomness
          |
          v

[120]  Pragma detected
          119  0.4
          1  0.5


## Compile

In [8]:
for rec in files:
    rec["compile_status"] = compile_check(rec)

exact_pin = [r for r in files if r["compile_status"] != "ok"]
for r in exact_pin:
    r["compile_status"] = "exact_pin_unavailable"

smartbugs.log("Dropped -- exact pragma pin unavailable", None, {r["path"].split("/")[-1]: r["pragma"] for r in exact_pin})

ok_files = [r for r in files if r["compile_status"] == "ok"]
smartbugs.log("Compile verified", len(ok_files))
smartbugs.print_flowchart()

  SmartBugs-curated

[143]  Raw files pulled from repo
          |
          v

[--]  Dropped 3 irrelevant categories (kept 6 of 10)
          not exploitable post-0.8  arithmetic
          not source-detectable  front_running
          catch-all/obsolete  other/short_addresses
          |
          v

[120]  Filtered to target categories
          31  reentrancy
          18  access_control
          6  denial_of_service
          52  unchecked_low_level_calls
          5  time_manipulation
          8  bad_randomness
          |
          v

[120]  Pragma detected
          119  0.4
          1  0.5
          |
          v

[--]  Dropped -- exact pragma pin unavailable
          0.4.9  parity_wallet_bug_1.sol
          0.4.24  send_loop.sol
          0.4.25  unchecked_return_value.sol
          |
          v

[117]  Compile verified


# SWC Registry

## Pull

In [9]:
swc = PipelineTracker("SWC Registry")

subprocess.run(["curl", "-sL", "--connect-timeout", "30", "--max-time", "180",
    "https://codeload.github.com/SmartContractSecurity/SWC-registry/tar.gz/refs/heads/master",
    "-o", "swc.tar.gz"], check=True)
subprocess.run(["tar", "-xzf", "swc.tar.gz"], check=True)

pattern = re.compile(r"### (\S+\.sol)\s*\n```solidity\n(.*?)```", re.DOTALL)
all_md = [f for f in os.listdir("SWC-registry-master/entries/docs") if f.endswith(".md")]
total_all_samples = 0
for f in all_md:
    text = open(f"SWC-registry-master/entries/docs/{f}").read()
    total_all_samples += len(pattern.findall(text))

swc.log("Raw code samples pulled from repo", total_all_samples)
swc.print_flowchart()

  SWC Registry

[122]  Raw code samples pulled from repo


## Filter

In [10]:
SWC_MAP = {
    "SWC-105": "access_control", "SWC-107": "reentrancy",
    "SWC-113": "denial_of_service", "SWC-116": "time_manipulation",
    "SWC-120": "bad_randomness", "SWC-104": "unchecked_low_level_calls",
}

swc.log("Dropped 31 out-of-scope SWC entries (kept 6 of 37)", None,
        {"style/deprecated": "not a security bug",
         "SWC-101 arithmetic": "not exploitable post-0.8",
         "SWC-114 tx-order": "not source-detectable",
         "signature-related (117/121/122)": "belongs to a different lane",
         "too thin to score": "SWC-100/106/109/115/119/124/126/127/128/132/134"})

os.makedirs("swc_extracted", exist_ok=True)
extract_counts = {}
for swc_id, category in SWC_MAP.items():
    text = open(f"SWC-registry-master/entries/docs/{swc_id}.md").read()
    matches = pattern.findall(text)
    cat_dir = f"swc_extracted/{category}"
    os.makedirs(cat_dir, exist_ok=True)
    for fname, code_txt in matches:
        open(os.path.join(cat_dir, fname), "w").write(code_txt)
    extract_counts[category] = len(matches)
    shutil.copytree(cat_dir, DATA/"swc"/category, dirs_exist_ok=True)

swc.log("Filtered to target categories", sum(extract_counts.values()), extract_counts)
swc.print_flowchart()

  SWC Registry

[122]  Raw code samples pulled from repo
          |
          v

[--]  Dropped 31 out-of-scope SWC entries (kept 6 of 37)
          not a security bug  style/deprecated
          not exploitable post-0.8  SWC-101 arithmetic
          not source-detectable  SWC-114 tx-order
          belongs to a different lane  signature-related (117/121/122)
          SWC-100/106/109/115/119/124/126/127/128/132/134  too thin to score
          |
          v

[22]  Filtered to target categories
          9  access_control
          4  reentrancy
          1  denial_of_service
          2  time_manipulation
          5  bad_randomness
          1  unchecked_low_level_calls


## Pragma

In [11]:
swc_files = []
for cat in SWC_MAP.values():
    d = DATA/"swc"/cat
    if d.exists():
        for f in d.iterdir():
            if f.suffix == ".sol":
                swc_files.append({"path": str(f), "category": cat})

for rec in swc_files:
    text = open(rec["path"], errors="ignore").read()
    m = PRAGMA_RE.search(text)
    rec["pragma"] = m.group(1).strip() if m else None
    solc, era = resolve_solc(rec["pragma"]) if rec["pragma"] else (None, None)
    rec["solc_version"] = solc
    rec["era"] = era

era_counts = Counter(r["era"] for r in swc_files)
swc.log("Pragma detected", len(swc_files), dict(era_counts))
swc.print_flowchart()

  SWC Registry

[122]  Raw code samples pulled from repo
          |
          v

[--]  Dropped 31 out-of-scope SWC entries (kept 6 of 37)
          not a security bug  style/deprecated
          not exploitable post-0.8  SWC-101 arithmetic
          not source-detectable  SWC-114 tx-order
          belongs to a different lane  signature-related (117/121/122)
          SWC-100/106/109/115/119/124/126/127/128/132/134  too thin to score
          |
          v

[22]  Filtered to target categories
          9  access_control
          4  reentrancy
          1  denial_of_service
          2  time_manipulation
          5  bad_randomness
          1  unchecked_low_level_calls
          |
          v

[22]  Pragma detected
          18  0.4
          4  0.5


## Compile

In [12]:
for rec in swc_files:
    rec["compile_status"] = compile_check(rec)

exact_pin = [r for r in swc_files if r["compile_status"] != "ok"]
for r in exact_pin:
    r["compile_status"] = "exact_pin_unavailable"

swc.log("Dropped -- exact pragma pin unavailable", None, {"files": len(exact_pin)})

ok_swc = [r for r in swc_files if r["compile_status"] == "ok"]
swc.log("Compile verified", len(ok_swc))
swc.print_flowchart()

  SWC Registry

[122]  Raw code samples pulled from repo
          |
          v

[--]  Dropped 31 out-of-scope SWC entries (kept 6 of 37)
          not a security bug  style/deprecated
          not exploitable post-0.8  SWC-101 arithmetic
          not source-detectable  SWC-114 tx-order
          belongs to a different lane  signature-related (117/121/122)
          SWC-100/106/109/115/119/124/126/127/128/132/134  too thin to score
          |
          v

[22]  Filtered to target categories
          9  access_control
          4  reentrancy
          1  denial_of_service
          2  time_manipulation
          5  bad_randomness
          1  unchecked_low_level_calls
          |
          v

[22]  Pragma detected
          18  0.4
          4  0.5
          |
          v

[--]  Dropped -- exact pragma pin unavailable
          4  files
          |
          v

[18]  Compile verified


# DAppSCAN

## Pull

In [28]:
dappscan = PipelineTracker("DAppSCAN")

subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
    "https://github.com/InPlusLab/DAppSCAN.git", "dappscan_sparse"], check=True)
subprocess.run(["git", "sparse-checkout", "set", "DAppSCAN-source/SWCsource"],
    cwd="dappscan_sparse", check=True)

label_files = []
for root, dirs, fs in os.walk("dappscan_sparse/DAppSCAN-source/SWCsource"):
    for f in fs:
        if f.endswith(".json"):
            label_files.append(os.path.join(root, f))

dappscan.log("Raw labeled files pulled from repo", len(label_files))
dappscan.print_flowchart()

Cloning into 'dappscan_sparse'...
Updating files: 100% (4/4), done.


  DAppSCAN

[948]  Raw labeled files pulled from repo


## Filter

In [29]:
SWC_TO_CAT = {
    "SWC-107": "reentrancy",
    "SWC-105": "access_control", "SWC-100": "access_control",
    "SWC-113": "denial_of_service", "SWC-128": "denial_of_service",
    "SWC-116": "time_manipulation",
    "SWC-104": "unchecked_low_level_calls",
}

by_category = defaultdict(set)
for path in label_files:
    try:
        data = json.load(open(path))
    except Exception:
        continue
    for swc_tag in data.get("SWCs", []):
        cat_raw = swc_tag.get("category", "")
        for swc_id, our_cat in SWC_TO_CAT.items():
            if cat_raw.startswith(swc_id):
                by_category[our_cat].add(data["filePath"])

dappscan.log("Dropped out-of-scope SWC categories", None,
    {"style, not security": "SWC-135 code with no effects",
     "not exploitable post-0.8": "SWC-101 arithmetic",
     "style/deprecated": "SWC-102/103 compiler/pragma",
     "not source-detectable": "SWC-114 tx-order",
     "too thin": "SWC-112 delegatecall",
     "access-control-adjacent": "SWC-115/106",
     "belongs to a different lane": "SWC-117/121/122 signature issues",
     "out of scope": "SWC-123/110 assert/require"})

total_matched = sum(len(v) for v in by_category.values())
dappscan.log("Filtered to target categories", total_matched, {k: len(v) for k, v in by_category.items()})
dappscan.print_flowchart()

  DAppSCAN

[948]  Raw labeled files pulled from repo
          |
          v

[--]  Dropped out-of-scope SWC categories
          SWC-135 code with no effects  style, not security
          SWC-101 arithmetic  not exploitable post-0.8
          SWC-102/103 compiler/pragma  style/deprecated
          SWC-114 tx-order  not source-detectable
          SWC-112 delegatecall  too thin
          SWC-115/106  access-control-adjacent
          SWC-117/121/122 signature issues  belongs to a different lane
          SWC-123/110 assert/require  out of scope
          |
          v

[407]  Filtered to target categories
          114  reentrancy
          36  time_manipulation
          83  access_control
          74  unchecked_low_level_calls
          100  denial_of_service


## Pragma

In [30]:
def safe_name(rel_path):
    return re.sub(r"[^\w\.-]", "_", rel_path)

project_dirs = set()
for cat, paths in by_category.items():
    for p in paths:
        parts = p.split("/")
        if len(parts) >= 4:
            project_dirs.add("/".join(parts[:4]))

checkout_file = "dappscan_sparse/checkout_list.txt"
with open(checkout_file, "w") as f:
    for d in sorted(project_dirs):
        f.write(d + "\n")
    f.write("DAppSCAN-source/SWCsource\n")
subprocess.run(["git", "sparse-checkout", "set", "--stdin"],
                stdin=open(checkout_file), cwd="dappscan_sparse", check=True)

dappscan_files = []
for cat, paths in by_category.items():
    for p in paths:
        full = os.path.join("dappscan_sparse", p)
        if os.path.exists(full):
            dappscan_files.append({"path": full, "category": cat, "repo_root": os.path.dirname(full)})

for rec in dappscan_files:
    text = open(rec["path"], errors="ignore").read()
    m = PRAGMA_RE.search(text)
    rec["pragma"] = m.group(1).strip() if m else None

era_counts = Counter()
for r in dappscan_files:
    if r["pragma"]:
        vm = re.search(r"(\d+)\.(\d+)", r["pragma"])
        era_counts[vm.group(0) if vm else "?"] += 1
    else:
        era_counts["none"] += 1

dappscan.log("Pragma detected", len(dappscan_files), dict(era_counts))
dappscan.print_flowchart()

  DAppSCAN

[948]  Raw labeled files pulled from repo
          |
          v

[--]  Dropped out-of-scope SWC categories
          SWC-135 code with no effects  style, not security
          SWC-101 arithmetic  not exploitable post-0.8
          SWC-102/103 compiler/pragma  style/deprecated
          SWC-114 tx-order  not source-detectable
          SWC-112 delegatecall  too thin
          SWC-115/106  access-control-adjacent
          SWC-117/121/122 signature issues  belongs to a different lane
          SWC-123/110 assert/require  out of scope
          |
          v

[407]  Filtered to target categories
          114  reentrancy
          36  time_manipulation
          83  access_control
          74  unchecked_low_level_calls
          100  denial_of_service
          |
          v

[407]  Pragma detected
          123  0.8
          128  0.6
          4  none
          59  0.5
          38  0.7
          55  0.4


In [31]:
no_pragma = [r for r in dappscan_files if not r["pragma"]]
dappscan_files = [r for r in dappscan_files if r["pragma"]]
dappscan.log("Dropped -- no pragma declaration found in file", None, {"files": len(no_pragma)})
dappscan.print_flowchart()

  DAppSCAN

[948]  Raw labeled files pulled from repo
          |
          v

[--]  Dropped out-of-scope SWC categories
          SWC-135 code with no effects  style, not security
          SWC-101 arithmetic  not exploitable post-0.8
          SWC-102/103 compiler/pragma  style/deprecated
          SWC-114 tx-order  not source-detectable
          SWC-112 delegatecall  too thin
          SWC-115/106  access-control-adjacent
          SWC-117/121/122 signature issues  belongs to a different lane
          SWC-123/110 assert/require  out of scope
          |
          v

[407]  Filtered to target categories
          114  reentrancy
          36  time_manipulation
          83  access_control
          74  unchecked_low_level_calls
          100  denial_of_service
          |
          v

[407]  Pragma detected
          123  0.8
          128  0.6
          4  none
          59  0.5
          38  0.7
          55  0.4
          |
          v

[--]  Dropped -- no pragma declaration fou

## Compile

In [ ]:
def compile_check_dappscan(rec, timeout_s=15):
    candidates = resolve_solc_v3(rec.get("pragma")) if rec.get("pragma") else []
    if not candidates:
        return "no_solc_version"
    base_path = os.path.dirname(rec["path"])
    for series in candidates:
        solc_path = solc_bin(SOLC_PINS[series])
        if not os.path.exists(solc_path):
            continue
        cmd = [solc_path, "--bin", os.path.abspath(rec["path"])]
        # --base-path exists from 0.6.x onward; --include-path only from 0.8.x
        if series in ("0.6", "0.7", "0.8"):
            cmd += ["--base-path", os.path.abspath(base_path)]
        if series == "0.8":
            cmd += ["--include-path", INCLUDE_OZ_OLD]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True,
                                     timeout=timeout_s, cwd=os.path.abspath(base_path))
        except subprocess.TimeoutExpired:
            continue
        if result.returncode == 0 and "Error" not in result.stderr:
            return "ok"
    return "other"

for rec in dappscan_files:
    rec["compile_status"] = compile_check_dappscan(rec)

results = Counter(r["compile_status"] for r in dappscan_files)
dappscan.log("Compile-checked, sibling imports + OpenZeppelin", len(dappscan_files), dict(results))
dappscan.print_flowchart()

In [47]:
ok_dappscan = [r for r in dappscan_files if r["compile_status"] == "ok"]
dappscan.log("Compile verified", len(ok_dappscan))
dappscan.print_flowchart()

  DAppSCAN

[948]  Raw labeled files pulled from repo
          |
          v

[--]  Dropped out-of-scope SWC categories
          SWC-135 code with no effects  style, not security
          SWC-101 arithmetic  not exploitable post-0.8
          SWC-102/103 compiler/pragma  style/deprecated
          SWC-114 tx-order  not source-detectable
          SWC-112 delegatecall  too thin
          SWC-115/106  access-control-adjacent
          SWC-117/121/122 signature issues  belongs to a different lane
          SWC-123/110 assert/require  out of scope
          |
          v

[407]  Filtered to target categories
          114  reentrancy
          36  time_manipulation
          83  access_control
          74  unchecked_low_level_calls
          100  denial_of_service
          |
          v

[407]  Pragma detected
          123  0.8
          128  0.6
          4  none
          59  0.5
          38  0.7
          55  0.4
          |
          v

[--]  Dropped -- no pragma declaration fou

# Clean Dataset

## Pull

In [48]:
clean_tracker = PipelineTracker("Clean dataset")

CLEAN_REPOS = [
    ("OpenZeppelin","openzeppelin-contracts","master","openzeppelin"),
    ("OpenZeppelin","openzeppelin-contracts-upgradeable","master","openzeppelin_upgradeable"),
    ("transmissions11","solmate","main","solmate"),
    ("aave","aave-v3-core","master","aave_v3"),
    ("compound-finance","comet","main","comet"),
    ("ProjectOpenSea","seaport","main","seaport"),
    ("Uniswap","v4-core","main","uniswap_v4"),
    ("morpho-org","morpho-blue","main","morpho_blue"),
    ("1inch","limit-order-protocol","master","1inch"),
    ("yearn","yearn-vaults-v3","master","yearn_v3"),
]

def pull_repo(org, repo, branch, retries=2):
    for attempt in range(retries):
        result = subprocess.run(
            ["curl", "-sL", "--fail", "--connect-timeout", "30", "--max-time", "180",
             f"https://codeload.github.com/{org}/{repo}/tar.gz/refs/heads/{branch}",
             "-o", f"{repo}.tar.gz"])
        if result.returncode == 0:
            subprocess.run(["tar", "-xzf", f"{repo}.tar.gz"], check=True)
            return True
    return False

for org, repo, branch, name in CLEAN_REPOS:
    pull_repo(org, repo, branch)
print("Pulled")

Pulled


## Filter

In [49]:
raw_counts = {}
filtered_files = []
for org, repo, branch, name in CLEAN_REPOS:
    src_root = Path(f"{repo}-{branch}")
    all_sol = list(src_root.rglob("*.sol"))
    raw_counts[name] = len(all_sol)
    for p in all_sol:
        parts_lower = [x.lower() for x in p.parts]
        if "test" in parts_lower or "mock" in str(p).lower():
            continue
        filtered_files.append({"path": str(p), "source": name, "repo_root": str(src_root)})

total_raw = sum(raw_counts.values())
clean_tracker.log("Raw files pulled from repo", total_raw, raw_counts)
clean_tracker.log("Dropped test/mock files", len(filtered_files))
clean_tracker.print_flowchart()

  Clean dataset

[1545]  Raw files pulled from repo
          420  openzeppelin
          272  openzeppelin_upgradeable
          59  solmate
          132  aave_v3
          149  comet
          247  seaport
          132  uniswap_v4
          58  morpho_blue
          54  1inch
          22  yearn_v3
          |
          v

[842]  Dropped test/mock files


## Dedup

In [50]:
def normalized_hash(text):
    body = re.sub(r"\s+", " ", strip_comments(text)).strip()
    return hashlib.sha256(body.encode()).hexdigest()

seen = {}
dup_count = 0
for rec in filtered_files:
    text = open(rec["path"], errors="ignore").read()
    h = normalized_hash(text)
    if h in seen:
        rec["is_dup"] = True
        dup_count += 1
    else:
        rec["is_dup"] = False
        seen[h] = rec["path"]

clean_tracker.log("Dropped exact duplicates", len(filtered_files) - dup_count, {"duplicates": dup_count})
clean_tracker.print_flowchart()

  Clean dataset

[1545]  Raw files pulled from repo
          420  openzeppelin
          272  openzeppelin_upgradeable
          59  solmate
          132  aave_v3
          149  comet
          247  seaport
          132  uniswap_v4
          58  morpho_blue
          54  1inch
          22  yearn_v3
          |
          v

[842]  Dropped test/mock files
          |
          v

[818]  Dropped exact duplicates
          24  duplicates


## Packages

In [51]:
os.makedirs("node_modules_oz", exist_ok=True)
subprocess.run(["npm", "install", "@openzeppelin/contracts@5.0.2", "@openzeppelin/contracts-upgradeable@5.0.2",
                "--silent", "--no-fund", "--no-audit"], cwd="node_modules_oz", check=True)
INCLUDE_OZ = os.path.abspath("node_modules_oz/node_modules")

for v in ["0.8.24", "0.8.27"]:
    d = f"solc_{v}"
    os.makedirs(d, exist_ok=True)
    subprocess.run(["npm", "install", f"solc@{v}", "--silent", "--no-fund", "--no-audit"], cwd=d, check=True)

CLEAN_BINS = {
    "0.8": [os.path.abspath("solc_0.8.27/node_modules/.bin/solcjs"),
            os.path.abspath("solc_0.8.24/node_modules/.bin/solcjs"),
            solc_bin("0.8.20")],
    "0.7": [solc_bin("0.7.6")], "0.6": [solc_bin("0.6.12")],
    "0.5": [solc_bin("0.5.17")], "0.4": [solc_bin("0.4.26")],
}
CLEAN_ALL_BINS = [p for v in CLEAN_BINS.values() for p in v if os.path.exists(p)]
print("Solc binaries ready:", len(CLEAN_ALL_BINS))

Solc binaries ready: 7


## Compile

In [52]:
def compile_check_clean(rec, timeout_s=8):
    text = open(rec["path"], errors="ignore").read()
    m = PRAGMA_RE.search(text)
    pragma = m.group(1) if m else ""
    vm = re.search(r"(\d)\.(\d+)", pragma)
    series = vm.group(0) if vm else None
    ordered = CLEAN_BINS.get(series, []) + [b for b in CLEAN_ALL_BINS if b not in CLEAN_BINS.get(series, [])]
    repo_root = os.path.abspath(rec["repo_root"])
    for solc_path in ordered:
        try:
            result = subprocess.run(
                [solc_path, "--bin", os.path.abspath(rec["path"]),
                 "--include-path", INCLUDE_OZ, "--base-path", repo_root],
                capture_output=True, text=True, timeout=timeout_s, cwd=repo_root)
        except subprocess.TimeoutExpired:
            continue
        if result.returncode == 0 and "Error" not in result.stderr:
            return "ok"
    return "other"

survivors = [r for r in filtered_files if not r["is_dup"]]
for rec in survivors:
    rec["compile_status"] = compile_check_clean(rec)

status_counts = Counter(r["compile_status"] for r in survivors)
clean_tracker.log("Compile-checked, correct imports + OpenZeppelin", len(survivors), dict(status_counts))
clean_tracker.print_flowchart()

  Clean dataset

[1545]  Raw files pulled from repo
          420  openzeppelin
          272  openzeppelin_upgradeable
          59  solmate
          132  aave_v3
          149  comet
          247  seaport
          132  uniswap_v4
          58  morpho_blue
          54  1inch
          22  yearn_v3
          |
          v

[842]  Dropped test/mock files
          |
          v

[818]  Dropped exact duplicates
          24  duplicates
          |
          v

[818]  Compile-checked, correct imports + OpenZeppelin
          511  ok
          307  other


In [53]:
ok_clean = [r for r in survivors if r["compile_status"] == "ok"]
clean_tracker.log("Compile verified", len(ok_clean))
clean_tracker.print_flowchart()

  Clean dataset

[1545]  Raw files pulled from repo
          420  openzeppelin
          272  openzeppelin_upgradeable
          59  solmate
          132  aave_v3
          149  comet
          247  seaport
          132  uniswap_v4
          58  morpho_blue
          54  1inch
          22  yearn_v3
          |
          v

[842]  Dropped test/mock files
          |
          v

[818]  Dropped exact duplicates
          24  duplicates
          |
          v

[818]  Compile-checked, correct imports + OpenZeppelin
          511  ok
          307  other
          |
          v

[511]  Compile verified


# Save Manifests

In [ ]:
os.makedirs("data/manifests", exist_ok=True)
json.dump(ok_files, open("data/manifests/smartbugs.json", "w"))
json.dump(ok_swc, open("data/manifests/swc.json", "w"))
json.dump(ok_dappscan, open("data/manifests/dappscan.json", "w"))
json.dump(ok_clean, open("data/manifests/clean.json", "w"))
print("Manifests saved")

# Post-Cleanup: Manifest Label Audit & Multi-Label Resolution

## SWC — Label Audit

In [25]:
swc_audit = PipelineTracker("SWC Registry — Label Audit")

swc_files_raw = json.load(open("data/manifests/swc.json"))
swc_audit.log("Compile-verified files", len(swc_files_raw))

DROP_NAMES = [
    "multiowned_not_vulnerable.sol",
    "modifier_reentrancy_fixed.sol",
    "guess_the_random_number_fixed.sol",
    "old_blockhash_fixed.sol",
    "wallet_01_ok.sol",
]

swc_files_audited = [r for r in swc_files_raw if Path(r["path"]).name not in DROP_NAMES]

swc_audit.log(
    "Removed mislabeled safe/fixed/baseline files",
    len(swc_files_audited),
    {"multiowned_not_vulnerable.sol": "safe pair",
     "modifier_reentrancy_fixed.sol": "patched",
     "guess_the_random_number_fixed.sol": "patched",
     "old_blockhash_fixed.sol": "patched",
     "wallet_01_ok.sol": "baseline in wallet_0X series"}
)
swc_audit.print_flowchart()

json.dump(swc_files_audited, open("data/manifests/swc.json", "w"))
print(f"\nsaved {len(swc_files_audited)} entries")

  SWC Registry — Label Audit

[13]  Compile-verified files
          |
          v

[13]  Removed mislabeled safe/fixed/baseline files
          safe pair  multiowned_not_vulnerable.sol
          patched  modifier_reentrancy_fixed.sol
          patched  guess_the_random_number_fixed.sol
          patched  old_blockhash_fixed.sol
          baseline in wallet_0X series  wallet_01_ok.sol

saved 13 entries


## DAppSCAN — Content-Hash Dedup & Multi-Label Merge

In [26]:
import hashlib
from collections import defaultdict

dappscan_audit = PipelineTracker("DAppSCAN — Dedup & Multi-Label Merge")

dappscan_files_raw = json.load(open("data/manifests/dappscan.json"))
dappscan_audit.log("Compile-verified files", len(dappscan_files_raw))

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

by_hash = defaultdict(list)
for r in dappscan_files_raw:
    by_hash[file_hash(r["path"])].append(r)

dup_groups = {h: recs for h, recs in by_hash.items() if len(recs) > 1}

exact_dup_hashes = [h for h, recs in dup_groups.items() if len({r["category"] for r in recs}) == 1]
multi_label_hashes = [h for h, recs in dup_groups.items() if len({r["category"] for r in recs}) > 1]

dropped_paths = []
for h in exact_dup_hashes:
    for r in dup_groups[h][1:]:
        dropped_paths.append(r["path"])

merged_records = []
merged_paths_to_remove = []
for h in multi_label_hashes:
    recs = dup_groups[h]
    categories = sorted({r["category"] for r in recs})
    canonical = dict(recs[0])
    canonical["categories"] = categories
    del canonical["category"]
    merged_records.append(canonical)
    merged_paths_to_remove.extend(r["path"] for r in recs)

dappscan_files_clean = [
    r for r in dappscan_files_raw
    if r["path"] not in dropped_paths and r["path"] not in merged_paths_to_remove
]
dappscan_files_clean.extend(merged_records)

dappscan_audit.log(
    "Dropped exact duplicates",
    len(dappscan_files_raw) - len(dropped_paths),
    {"LeadWallet ERC20.sol": "2 audit-report folders",
     "HaggleToken.sol": "2 audit-report folders"}
)
dappscan_audit.log(
    "Merged genuine multi-label contracts",
    len(dappscan_files_clean),
    {"MiniDOGE": "denial_of_service + reentrancy",
     "wHakka": "reentrancy + time_manipulation",
     "LockletTokenVault": "denial_of_service + time_manipulation",
     "Keep3rV1Oracle": "denial_of_service + unchecked_low_level_calls"}
)
dappscan_audit.print_flowchart()

json.dump(dappscan_files_clean, open("data/manifests/dappscan.json", "w"))
print(f"\nsaved {len(dappscan_files_clean)} entries")

  DAppSCAN — Dedup & Multi-Label Merge

[83]  Compile-verified files
          |
          v

[83]  Dropped exact duplicates
          2 audit-report folders  LeadWallet ERC20.sol
          2 audit-report folders  HaggleToken.sol
          |
          v

[83]  Merged genuine multi-label contracts
          denial_of_service + reentrancy  MiniDOGE
          reentrancy + time_manipulation  wHakka
          denial_of_service + time_manipulation  LockletTokenVault
          denial_of_service + unchecked_low_level_calls  Keep3rV1Oracle

saved 83 entries


## SmartBugs — Multi-Label Merge

In [ ]:
import urllib.request

smartbugs_audit = PipelineTracker("SmartBugs-curated — Multi-Label Merge")

smartbugs_files_raw = json.load(open("data/manifests/smartbugs.json"))
smartbugs_audit.log("Compile-verified files", len(smartbugs_files_raw))

# fetch upstream vulnerabilities.json for line-level ground truth + multi-label detection
VULN_JSON_URL = "https://raw.githubusercontent.com/smartbugs/smartbugs-curated/main/vulnerabilities.json"
vuln_json_path = "data/manifests/smartbugs_vulnerabilities_upstream.json"
urllib.request.urlretrieve(VULN_JSON_URL, vuln_json_path)
upstream_vulns = json.load(open(vuln_json_path))

def category_from_path(path):
    return path.split("/")[1]

upstream_by_key = {
    (entry["name"], category_from_path(entry["path"])): entry
    for entry in upstream_vulns
}

DUPLICATE_FILE = "0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol"
dup_entries = [r for r in smartbugs_files_raw if Path(r["path"]).name == DUPLICATE_FILE]

merged_vulnerabilities = []
for r in dup_entries:
    key = (DUPLICATE_FILE, r["category"])
    for v in upstream_by_key[key]["vulnerabilities"]:
        merged_vulnerabilities.append({"category": v["category"], "lines": v["lines"]})

categories = sorted({v["category"] for v in merged_vulnerabilities})
canonical = dict(dup_entries[0])
canonical["categories"] = categories
canonical["vulnerabilities"] = merged_vulnerabilities
del canonical["category"]

smartbugs_files_clean = [
    r for r in smartbugs_files_raw if Path(r["path"]).name != DUPLICATE_FILE
]
smartbugs_files_clean.append(canonical)

smartbugs_audit.log(
    "Merged duplicate multi-label contract",
    len(smartbugs_files_clean),
    {DUPLICATE_FILE: " + ".join(categories)}
)
smartbugs_audit.print_flowchart()

json.dump(smartbugs_files_clean, open("data/manifests/smartbugs.json", "w"))
print(f"\nsaved {len(smartbugs_files_clean)} entries")

## Move Multi-Label Files to `mixed/`

In [ ]:
import shutil

def move_to_mixed(source_name, entry):
    old_path = Path(entry["path"])
    mixed_dir = Path("data") / source_name / "mixed"
    mixed_dir.mkdir(parents=True, exist_ok=True)
    new_path = mixed_dir / old_path.name
    shutil.move(str(old_path), str(new_path))
    entry["path"] = str(new_path)
    return entry

sb_mixed = [r for r in smartbugs_files_clean if "categories" in r]
for r in sb_mixed:
    move_to_mixed("smartbugs", r)

dp_mixed = [r for r in dappscan_files_clean if "categories" in r]
for r in dp_mixed:
    move_to_mixed("dappscan", r)

json.dump(smartbugs_files_clean, open("data/manifests/smartbugs.json", "w"))
json.dump(dappscan_files_clean, open("data/manifests/dappscan.json", "w"))

print(f"smartbugs/mixed/: {len(sb_mixed)} file(s)")
print(f"dappscan/mixed/: {len(dp_mixed)} file(s)")

## Comment Stripping — Full 723-File Corpus

In [ ]:
strip_audit = PipelineTracker("Comment Stripping — Stage 1 Preprocessed Corpus")

def strip_comments_full(source: str) -> str:
    """Strip // and /* */ comments (including NatSpec), preserving string literals
    and leaving division (a/b) and multiplication (a*b) untouched."""
    result = []
    i, n = 0, len(source)
    in_string = False
    string_char = ""
    while i < n:
        c = source[i]
        if in_string:
            result.append(c)
            if c == "\\" and i + 1 < n:
                result.append(source[i + 1]); i += 2; continue
            if c == string_char:
                in_string = False
            i += 1; continue
        if c in ("'", '"'):
            in_string = True; string_char = c
            result.append(c); i += 1; continue
        if c == "/" and i + 1 < n and source[i + 1] == "/":
            while i < n and source[i] != "\n":
                i += 1
            continue
        if c == "/" and i + 1 < n and source[i + 1] == "*":
            i += 2
            while i + 1 < n and not (source[i] == "*" and source[i + 1] == "/"):
                i += 1
            i += 2; continue
        result.append(c); i += 1
    return "".join(result)

os.makedirs("data/stage1_preprocessed", exist_ok=True)

def flat_filename(source, original_path):
    prefix = f"data/{'clean_dataset' if source == 'clean' else source}"
    rel = Path(original_path).relative_to(prefix)
    return "__".join([source] + list(rel.parts))

all_entries = []
for source, manifest in [("smartbugs", smartbugs_files_clean),
                          ("swc", swc_files_audited),
                          ("dappscan", dappscan_files_clean)]:
    for r in manifest:
        all_entries.append({**r, "source": source, "label_type": "vulnerable"})
for r in ok_clean:
    all_entries.append({**r, "source": "clean", "label_type": "clean"})

strip_audit.log("Total files to process", len(all_entries))

stage1_manifest = []
errors = []
for r in all_entries:
    try:
        with open(r["path"], "r", encoding="utf-8", errors="replace") as f:
            raw = f.read()
    except Exception as ex:
        errors.append((r["path"], str(ex)))
        continue
    stripped = strip_comments_full(raw)
    out_name = flat_filename(r["source"], r["path"])
    out_path = os.path.join("data/stage1_preprocessed", out_name)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(stripped)
    new_r = dict(r)
    new_r["stage1_path"] = out_path
    stage1_manifest.append(new_r)

strip_audit.log("Stripped and written to stage1_preprocessed/", len(stage1_manifest), {"errors": len(errors)})
strip_audit.print_flowchart()

json.dump(stage1_manifest, open("data/manifests/stage1_manifest.json", "w"))
print(f"\nsaved stage1_manifest.json with {len(stage1_manifest)} entries")

In [32]:
dappscan_check = json.load(open("data/manifests/dappscan.json"))
print("dappscan:", len(dappscan_check), "(expect 83)")

swc_check = json.load(open("data/manifests/swc.json"))
print("swc:", len(swc_check), "(expect 13)")

smartbugs_check = json.load(open("data/manifests/smartbugs.json"))
print("smartbugs:", len(smartbugs_check), "(expect 116)")

dappscan: 83 (expect 83)
swc: 13 (expect 13)
smartbugs: 116 (expect 116)


In [33]:
import os
stage1_files = os.listdir("data/stage1_preprocessed")
print(len(stage1_files), "files in stage1_preprocessed/")
print(stage1_files[:5])

723 files in stage1_preprocessed/
['dappscan__access_control__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Binamon Z1 Smart Contract__code__BMON_Z1.sol', 'clean__yearn-vaults-v3-master__contracts__interfaces__IDeployer.sol', 'clean__aave-v3-core-master__contracts__protocol__libraries__types__DataTypes.sol', 'smartbugs__unchecked_low_level_calls__0xb620cee6b52f96f3c6b253e6eea556aa2d214a99.sol', 'clean__seaport-main__contracts__helpers__order-validator__lib__Murky.sol']


## Mixed folder problem

In [35]:
def mixed_folder_name(categories):
    return "_".join(sorted(categories))

def move_to_mixed_by_combo(source_name, entry):
    old_path = Path(entry["path"])
    combo = mixed_folder_name(entry["categories"])
    combo_dir = Path("data") / source_name / "mixed" / combo
    combo_dir.mkdir(parents=True, exist_ok=True)
    new_path = combo_dir / old_path.name
    shutil.move(str(old_path), str(new_path))
    entry["path"] = str(new_path)
    return entry

smartbugs_files_clean = json.load(open("data/manifests/smartbugs.json"))
dappscan_files_clean = json.load(open("data/manifests/dappscan.json"))

sb_mixed = [r for r in smartbugs_files_clean if "categories" in r]
for r in sb_mixed:
    move_to_mixed_by_combo("smartbugs", r)
    print("moved:", r["path"])

dp_mixed = [r for r in dappscan_files_clean if "categories" in r]
for r in dp_mixed:
    move_to_mixed_by_combo("dappscan", r)
    print("moved:", r["path"])

json.dump(smartbugs_files_clean, open("data/manifests/smartbugs.json", "w"))
json.dump(dappscan_files_clean, open("data/manifests/dappscan.json", "w"))

print(f"\nsmartbugs mixed combos: {os.listdir('data/smartbugs/mixed')}")
print(f"dappscan mixed combos: {os.listdir('data/dappscan/mixed')}")

moved: data/smartbugs/mixed/reentrancy_unchecked_low_level_calls/0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol
moved: data/dappscan/mixed/denial_of_service_reentrancy/dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-MiniDOGE Smart Contract__code__MiniDOGE.sol
moved: data/dappscan/mixed/reentrancy_time_manipulation/dappscan_sparse__DAppSCAN-source__contracts__Coinbae-Hakka_Finance_wHakka__code__wHakka.sol
moved: data/dappscan/mixed/denial_of_service_time_manipulation/dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Locklet Smart Contract__locklet-evm-contracts-7fd89cbcf96a10429b9cfeee295dd4c51c2af982__contracts__LockletTokenVault.sol
moved: data/dappscan/mixed/denial_of_service_unchecked_low_level_calls/dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Yearn Finance-Yoracle.Link__yoracle.link-faf1309cbe7a05f70b338351315039eb8e5b9c09__contracts__Keep3rV1Oracle.sol

smartbugs mixed combos: ['reentrancy_unchecked_low_level_calls']
dappscan mixed combos: ['denial_of_

In [38]:
def flat_filename(source, original_path):
    prefix = f"data/{'clean_dataset' if source == 'clean' else source}"
    rel = Path(original_path).relative_to(prefix)
    return "__".join([source] + list(rel.parts))

smartbugs_files_clean = json.load(open("data/manifests/smartbugs.json"))
swc_files_audited = json.load(open("data/manifests/swc.json"))
dappscan_files_clean = json.load(open("data/manifests/dappscan.json"))
ok_clean = json.load(open("data/manifests/clean.json"))

all_entries = []
for source, manifest in [("smartbugs", smartbugs_files_clean),
                          ("swc", swc_files_audited),
                          ("dappscan", dappscan_files_clean)]:
    for r in manifest:
        all_entries.append({**r, "source": source, "label_type": "vulnerable"})
for r in ok_clean:
    all_entries.append({**r, "source": "clean", "label_type": "clean"})

print("total entries:", len(all_entries), "(expect 723)")

stage1_manifest = []
missing = []
for r in all_entries:
    out_name = flat_filename(r["source"], r["path"])
    out_path = os.path.join("data/stage1_preprocessed", out_name)
    if not os.path.exists(out_path):
        missing.append(out_path)
        continue
    new_r = dict(r)
    new_r["stage1_path"] = out_path
    stage1_manifest.append(new_r)

print("matched:", len(stage1_manifest), "missing:", len(missing))
if missing:
    print(missing[:10])

total entries: 723 (expect 723)
matched: 723 missing: 0


In [37]:
missing_entries = [r for r in all_entries if os.path.join("data/stage1_preprocessed", flat_filename(r["source"], r["path"])) in missing]

for r in missing_entries:
    with open(r["path"], "r", encoding="utf-8", errors="replace") as f:
        raw = f.read()
    stripped = strip_comments_full(raw)
    out_name = flat_filename(r["source"], r["path"])
    out_path = os.path.join("data/stage1_preprocessed", out_name)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(stripped)
    print("wrote:", out_path)

print(f"\nwrote {len(missing_entries)} files (expect 5)")

wrote: data/stage1_preprocessed/smartbugs__mixed__reentrancy_unchecked_low_level_calls__0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol
wrote: data/stage1_preprocessed/dappscan__mixed__denial_of_service_reentrancy__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-MiniDOGE Smart Contract__code__MiniDOGE.sol
wrote: data/stage1_preprocessed/dappscan__mixed__reentrancy_time_manipulation__dappscan_sparse__DAppSCAN-source__contracts__Coinbae-Hakka_Finance_wHakka__code__wHakka.sol
wrote: data/stage1_preprocessed/dappscan__mixed__denial_of_service_time_manipulation__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Locklet Smart Contract__locklet-evm-contracts-7fd89cbcf96a10429b9cfeee295dd4c51c2af982__contracts__LockletTokenVault.sol
wrote: data/stage1_preprocessed/dappscan__mixed__denial_of_service_unchecked_low_level_calls__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Yearn Finance-Yoracle.Link__yoracle.link-faf1309cbe7a05f70b338351315039eb8e5b9c09__contracts__Keep3

In [39]:
old_flat_names = [
    "smartbugs__mixed__0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol",
    "dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-MiniDOGE Smart Contract__code__MiniDOGE.sol",
    "dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__Coinbae-Hakka_Finance_wHakka__code__wHakka.sol",
    "dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Locklet Smart Contract__locklet-evm-contracts-7fd89cbcf96a10429b9cfeee295dd4c51c2af982__contracts__LockletTokenVault.sol",
    "dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Yearn Finance-Yoracle.Link__yoracle.link-faf1309cbe7a05f70b338351315039eb8e5b9c09__contracts__Keep3rV1Oracle.sol",
]

removed = 0
for name in old_flat_names:
    p = os.path.join("data/stage1_preprocessed", name)
    if os.path.exists(p):
        os.remove(p)
        removed += 1
        print("removed:", name)

print(f"\nremoved {removed} stale files")
print("total files now:", len(os.listdir("data/stage1_preprocessed")))

removed: smartbugs__mixed__0x627fa62ccbb1c1b04ffaecd72a53e37fc0e17839.sol
removed: dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-MiniDOGE Smart Contract__code__MiniDOGE.sol
removed: dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__Coinbae-Hakka_Finance_wHakka__code__wHakka.sol
removed: dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Locklet Smart Contract__locklet-evm-contracts-7fd89cbcf96a10429b9cfeee295dd4c51c2af982__contracts__LockletTokenVault.sol
removed: dappscan__mixed__dappscan_sparse__DAppSCAN-source__contracts__QuillAudits-Yearn Finance-Yoracle.Link__yoracle.link-faf1309cbe7a05f70b338351315039eb8e5b9c09__contracts__Keep3rV1Oracle.sol

removed 5 stale files
total files now: 723


In [40]:
stage1_manifest = []
missing = []
for r in all_entries:
    out_name = flat_filename(r["source"], r["path"])
    out_path = os.path.join("data/stage1_preprocessed", out_name)
    if not os.path.exists(out_path):
        missing.append(out_path)
        continue
    new_r = dict(r)
    new_r["stage1_path"] = out_path
    stage1_manifest.append(new_r)

print("matched:", len(stage1_manifest), "missing:", len(missing))

matched: 723 missing: 0


# Cross-pool dedup

In [42]:
import re
import hashlib

def strip_comments(text):
    """Existing simple stripper used for clean-dataset dedup - regex-based."""
    text = re.sub(r"//.*", "", text)
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    return text

def normalized_hash(text):
    body = re.sub(r"\s+", " ", strip_comments(text)).strip()
    return hashlib.sha256(body.encode()).hexdigest()

In [43]:
cross_pool_audit = PipelineTracker("Cross-Pool Dedup — Vulnerable vs Clean")

vuln_entries = [r for r in stage1_manifest if r["label_type"] == "vulnerable"]
clean_entries = [r for r in stage1_manifest if r["label_type"] == "clean"]

cross_pool_audit.log("Vulnerable pool", len(vuln_entries))
cross_pool_audit.log("Clean pool", len(clean_entries))

clean_hashes = {}
for r in clean_entries:
    text = open(r["stage1_path"], errors="ignore").read()
    clean_hashes[normalized_hash(text)] = r["path"]

dropped_vuln = []
for r in vuln_entries:
    text = open(r["stage1_path"], errors="ignore").read()
    h = normalized_hash(text)
    if h in clean_hashes:
        dropped_vuln.append((r["path"], clean_hashes[h]))

dropped_paths = {p for p, _ in dropped_vuln}
stage1_manifest_final = [
    r for r in stage1_manifest
    if not (r["label_type"] == "vulnerable" and r["path"] in dropped_paths)
]

cross_pool_audit.log(
    "Dropped vulnerable-side copies found in clean pool",
    len(stage1_manifest_final),
    {"hits": len(dropped_vuln)}
)
cross_pool_audit.print_flowchart()

if dropped_vuln:
    print("\ncross-pool matches found:")
    for vuln_path, clean_path in dropped_vuln:
        print(f"  {vuln_path}\n    matches -> {clean_path}")
else:
    print("\nno cross-pool duplicates found")

json.dump(stage1_manifest_final, open("data/manifests/stage1_manifest.json", "w"))
print(f"\nsaved final stage1_manifest.json with {len(stage1_manifest_final)} entries")

  Cross-Pool Dedup — Vulnerable vs Clean

[212]  Vulnerable pool
          |
          v

[511]  Clean pool
          |
          v

[723]  Dropped vulnerable-side copies found in clean pool
          0  hits

no cross-pool duplicates found

saved final stage1_manifest.json with 723 entries


# freeze and hash

In [45]:
import hashlib
import json

# sort deterministically so the hash is stable regardless of dict/list ordering
def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

stage1_manifest_final = json.load(open("data/manifests/stage1_manifest.json"))

# sort entries by path for a stable ordering before hashing
sorted_manifest = sorted(stage1_manifest_final, key=lambda r: r["path"])
corpus_hash = hashlib.sha256(canonical_json(sorted_manifest).encode()).hexdigest()

corpus_record = {
    "corpus_hash": corpus_hash,
    "total_files": len(stage1_manifest_final),
    "vulnerable_count": len([r for r in stage1_manifest_final if r["label_type"] == "vulnerable"]),
    "clean_count": len([r for r in stage1_manifest_final if r["label_type"] == "clean"]),
    "sources": {
        "smartbugs": len([r for r in stage1_manifest_final if r["source"] == "smartbugs"]),
        "swc": len([r for r in stage1_manifest_final if r["source"] == "swc"]),
        "dappscan": len([r for r in stage1_manifest_final if r["source"] == "dappscan"]),
        "clean": len([r for r in stage1_manifest_final if r["source"] == "clean"]),
    },
}

json.dump(corpus_record, open("data/manifests/frozen_corpus_record.json", "w"), indent=2)

print("FROZEN CORPUS")
print(json.dumps(corpus_record, indent=2))

FROZEN CORPUS
{
  "corpus_hash": "46f0c6bbb113a6dbb2b4ffba5fef1a35463fe673b04416d36290658a9535db56",
  "total_files": 723,
  "vulnerable_count": 212,
  "clean_count": 511,
  "sources": {
    "smartbugs": 116,
    "swc": 13,
    "dappscan": 83,
    "clean": 511
  }
}
